# 2 — Multilayer Perceptron (2019 judgement, same tools)

## What this notebook is

**Column 2 of the retrospective.** The same 8×8 digits, the same NumPy, the
same scikit-learn — the network done with judgement I did not have in 2019.

Everything here was available in 2019. Adam (2014), He initialisation (2010),
dropout (2014), early stopping (1990s), learning-rate schedules (decades).
**This is a discipline gap, not a tooling gap** — and for a from-scratch
network the discipline *is* the work.

## The changes from column 1

| # | Change | Possible in 2019? |
|---|---|---|
| 1 | Keep the finite-difference gradient check as a **test**, not a one-off | Yes |
| 2 | Adam instead of plain SGD | Yes — published 2014 |
| 3 | Learning-rate schedule | Yes |
| 4 | L2 regularisation, with the strength chosen on a validation split | Yes |
| 5 | Early stopping on a held-out validation set | Yes |
| 6 | Proper weight initialisation ablation | Yes |
| 7 | Report per-class performance, not just accuracy | Yes |

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 20260929
rng = np.random.default_rng(SEED)

digits = load_digits()
X = digits.data.astype(float) / 16.0
y = digits.target

# Three-way split this time: training, validation, and a test set that nothing
# touches. Column 1 used two ways, and tuned nothing, so it did not need the
# third — but once early stopping and regularisation strength enter, the
# validation set becomes load-bearing.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=SEED + 1, stratify=y_temp
)
print(f"train {len(X_train)} | validation {len(X_val)} | test {len(X_test)}")
print(f"test positive-per-class balance: {np.bincount(y_test).tolist()}")


def one_hot(labels, n_classes=10):
    out = np.zeros((len(labels), n_classes))
    out[np.arange(len(labels)), labels] = 1.0
    return out


Y_train, Y_val, Y_test = one_hot(y_train), one_hot(y_val), one_hot(y_test)

## The network, with everything the 2019 version lacked

In [ ]:
def relu(z):
    return np.maximum(0, z)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def init_weights(layers, rng, mode="he"):
    """Weight initialisation, with the alternatives kept for the ablation."""
    Ws, bs = [], []
    for i in range(len(layers) - 1):
        n_in, n_out = layers[i], layers[i + 1]
        if mode == "he":
            W = rng.normal(0, np.sqrt(2.0 / n_in), (n_in, n_out))
        elif mode == "xavier":
            W = rng.normal(0, np.sqrt(1.0 / n_in), (n_in, n_out))
        elif mode == "zeros":
            W = np.zeros((n_in, n_out))
        elif mode == "small":
            W = rng.normal(0, 0.01, (n_in, n_out))
        bs.append(np.zeros(n_out))
        Ws.append(W)
    return Ws, bs


def forward(Xb, Ws, bs, dropout_mask=None, p_drop=0.0, mask_rng=None):
    """Forward pass with optional dropout on hidden layers.

    Args:
        mask_rng: Generator used to draw dropout masks. Pass an explicit
            generator when the mask must be identical across calls — seeding
            ``np.random`` does not affect a ``Generator``, and a mask that
            changes between forward passes makes the numerical gradient
            meaningless.
    """
    activations = [Xb]
    pre_activations = []
    masks = []
    n_layers = len(Ws)

    for i in range(n_layers):
        z = activations[-1] @ Ws[i] + bs[i]
        pre_activations.append(z)

        if i == n_layers - 1:
            a = softmax(z)
        else:
            a = relu(z)
            if dropout_mask is not None and p_drop > 0:
                source = mask_rng if mask_rng is not None else rng
                mask = (source.random(a.shape) > p_drop) / (1 - p_drop)
                a = a * mask
                masks.append(mask)
            else:
                masks.append(None)
        activations.append(a)

    return activations[-1], activations, pre_activations, masks


def cross_entropy(probs, Y):
    return -np.sum(Y * np.log(np.clip(probs, 1e-12, None))) / probs.shape[0]


def backward(Xb, Y, Ws, bs, acts, pre, masks, p_drop=0.0):
    """Backpropagation with dropout masking applied to the weight gradients."""
    n = Xb.shape[0]
    dWs = [None] * len(Ws)
    dbs = [None] * len(bs)

    delta = (softmax(pre[-1]) - Y) / n
    dWs[-1] = acts[-2].T @ delta
    dbs[-1] = delta.sum(axis=0)

    for i in range(len(Ws) - 2, -1, -1):
        delta = delta @ Ws[i + 1].T
        # Dropout is an identity on the backward pass: the mask that was
        # applied in forward must be applied to the gradient flowing back
        # through the same units, or the mask and the gradient disagree.
        if masks[i] is not None and p_drop > 0:
            delta = delta * masks[i]
        delta = delta * (pre[i] > 0)
        dWs[i] = acts[i].T @ delta
        dbs[i] = delta.sum(axis=0)

    return dWs, dbs


print("network defined with:", ", ".join(["He init", "dropout", "Adam", "early stopping"]))

## Change 1 — Keep the gradient check as a reusable test

Column 1 ran the check once, by hand, on a sample of weights. That verifies the
code as written. What is needed is a check that runs on *any* weights, so a
later edit cannot silently break it.

In [ ]:
def numerical_gradient(Xb, Y, Ws, bs, eps=1e-6, rng_check=25, seed=SEED):
    """Central-difference gradient at a random sample of parameters.

    Returns ``(values, positions)`` where ``values[i]`` is the numerical
    gradient for layer ``i`` restricted to the sampled positions. Checking
    every parameter costs one forward pass per parameter; sampling gives the
    same protection against a wrong formula for a fraction of the cost.

    The positions must be carried alongside the values: a full-size array with
    only the sampled entries filled would compare checked entries against
    *unchecked zeros*, which reads as a huge error and is meaningless.
    """
    r = np.random.default_rng(seed)
    values = [np.zeros(min(rng_check, W.size)) for W in Ws]
    positions = []
    for li, W in enumerate(Ws):
        flat_idx = r.choice(W.size, size=min(rng_check, W.size), replace=False)
        positions.append(flat_idx)

    for li, flat_idx in enumerate(positions):
        flat = Ws[li].ravel()          # a view: C-contiguous, so writes land
        for j, f in enumerate(flat_idx):
            orig = flat[f]
            flat[f] = orig + eps
            lp = cross_entropy(forward(Xb, Ws, bs)[0], Y)
            flat[f] = orig - eps
            lm = cross_entropy(forward(Xb, Ws, bs)[0], Y)
            flat[f] = orig
            values[li][j] = (lp - lm) / (2 * eps)

    return values, positions


def check_gradients(Xb, Y, layers, seed=SEED, tol=1e-6, n_check=25):
    """Assert analytic and numerical gradients agree. Returns the max error."""
    r = np.random.default_rng(seed)
    Ws, bs = init_weights(layers, r)

    probs, acts, pre, masks = forward(Xb, Ws, bs)
    dWs, _ = backward(Xb, Y, Ws, bs, acts, pre, masks)
    num_values, positions = numerical_gradient(Xb, Y, Ws, bs,
                                               rng_check=n_check, seed=seed)

    # Compare ONLY the positions that were actually checked.
    err = 0.0
    for li, flat_idx in enumerate(positions):
        analytic = dWs[li].ravel()[flat_idx]
        err = max(err, float(np.abs(analytic - num_values[li]).max()))

    status = "PASS" if err < tol else "FAIL"
    print(f"  {status}  max |analytic - numerical| = {err:.3e}")
    return err

In [ ]:
Xc, Yc = X_train[:32], Y_train[:32]
print("gradient check (sampled parameters, 5 different weight draws):")
worst = 0.0
for seed in range(5):
    worst = max(worst, check_gradients(Xc, Yc, [64, 32, 10], seed=SEED + seed))
print()
print(f"worst error across all draws: {worst:.3e}")

In [ ]:
# The same check with dropout active, which is a different code path.
#
# The subtlety: the numerical gradient perturbs a weight and re-runs forward.
# If the dropout mask is redrawn each time, the two forward passes differ in
# two places — the perturbed weight AND the mask — and the finite difference
# measures mask noise. The mask must be held fixed across all three passes
# (analytic, +eps, -eps).
print()
print("gradient check with dropout active (p = 0.5), mask held fixed:")

r = np.random.default_rng(SEED)
Ws, bs = init_weights([64, 32, 10], r)

mask_rng = np.random.default_rng(SEED + 99)
_probs, _acts, _pre, _masks = forward(Xc, Ws, bs, dropout_mask=True, p_drop=0.5,
                                      mask_rng=mask_rng)
dW_drop, _ = backward(Xc, Yc, Ws, bs, _acts, _pre, _masks, p_drop=0.5)

errs = []
for li, W in enumerate(Ws):
    flat = W.ravel()
    for f in range(min(8, W.size)):
        orig = flat[f]
        losses = []
        for eps in (1e-6, -1e-6):
            flat[f] = orig + eps
            # Same seed -> same mask on every call.
            losses.append(cross_entropy(
                forward(Xc, Ws, bs, dropout_mask=True, p_drop=0.5,
                        mask_rng=np.random.default_rng(SEED + 99))[0], Yc))
            flat[f] = orig
        num = (losses[0] - losses[1]) / 2e-6
        errs.append(abs(dW_drop[li].ravel()[f] - num))

dropout_err = max(errs)
print(f"  max |analytic - numerical| = {dropout_err:.3e}  ({len(errs)} parameters sampled)")
if dropout_err < 1e-6:
    print("  PASS — the dropout backward path applies the same mask as forward.")
else:
    print("  FAIL — the mask applied in backward differs from the one in forward.")

## Change 2, 3, 4, 5 — Adam, a schedule, L2, and early stopping

In [ ]:
class Adam:
    """Adam optimiser: momentum plus per-parameter adaptive step sizes."""

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0):
        self.lr, self.beta1, self.beta2 = lr, beta1, beta2
        self.eps, self.weight_decay = eps, weight_decay
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            if self.weight_decay:
                g = g + self.weight_decay * p
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def train_network(Xtr, Ytr, Xva, Yva, layers, epochs=120, lr=1e-3,
                  batch_size=32, weight_decay=0.0, p_drop=0.0, patience=20,
                  seed=SEED, verbose=True, label="", init_mode="he"):
    """Train with Adam, cosine LR decay, L2 and early stopping.

    Returns the best weights by validation loss, the full history, and the
    epoch at which training stopped.
    """
    r = np.random.default_rng(seed)
    global rng
    saved_rng = rng
    rng = np.random.default_rng(seed + 7)

    Ws, bs = init_weights(layers, r, mode=init_mode)
    params = Ws + bs
    opt = Adam(params, lr=lr, weight_decay=weight_decay)

    n = Xtr.shape[0]
    best_val = np.inf
    best_Ws = [w.copy() for w in Ws]
    best_bs = [b.copy() for b in bs]
    best_epoch = 0
    wait = 0

    train_hist, val_hist = [], []

    for epoch in range(epochs):
        # Cosine decay: halves the step size over the run, which is a schedule
        # rather than a constant, and materially improves the final loss.
        lr_now = 0.5 * lr * (1 + np.cos(np.pi * epoch / epochs))

        order = r.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            xb, yb = Xtr[idx], Ytr[idx]
            probs, acts, pre, masks = forward(xb, Ws, bs, dropout_mask=True, p_drop=p_drop)
            dWs, dbs = backward(xb, yb, Ws, bs, acts, pre, masks, p_drop=p_drop)
            opt.step(params, dWs + dbs)

        tr_loss = cross_entropy(forward(Xtr, Ws, bs)[0], Ytr)
        va_loss = cross_entropy(forward(Xva, Ws, bs)[0], Yva)
        train_hist.append(tr_loss)
        val_hist.append(va_loss)

        if va_loss < best_val - 1e-5:
            best_val, best_epoch, wait = va_loss, epoch, 0
            best_Ws = [w.copy() for w in Ws]
            best_bs = [b.copy() for b in bs]
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"  early stop at epoch {epoch + 1} "
                          f"(best epoch {best_epoch + 1})")
                break

    rng = saved_rng
    return (best_Ws, best_bs), (train_hist, val_hist), best_epoch + 1

In [ ]:
LAYERS = [64, 32, 10]

print("training with Adam, cosine decay, L2, early stopping:")
t0 = time.time()
(best_W, best_b), (tr_hist, va_hist), best_epoch = train_network(
    X_train, Y_train, X_val, Y_val, LAYERS,
    epochs=120, lr=1e-3, weight_decay=1e-4, p_drop=0.2, patience=20,
)
t_adam = time.time() - t0
print(f"trained in {t_adam:.1f}s, stopped at epoch {best_epoch}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
axes[0].plot(tr_hist, color="#4C72B0", lw=1.5, label="train")
axes[0].plot(va_hist, color="#C44E52", lw=1.5, label="validation")
axes[0].axvline(best_epoch - 1, ls="--", color="black", label="best epoch")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Train and validation loss")
axes[0].legend()

axes[1].plot(tr_hist, color="#4C72B0", lw=1.5, label="train")
axes[1].plot(va_hist, color="#C44E52", lw=1.5, label="validation")
axes[1].set_xlabel("epoch")
axes[1].set_yscale("log")
axes[1].set_title("Same, log scale — the generalisation gap")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
pred = forward(X_test, best_W, best_b)[0].argmax(axis=1)
acc_adam = (pred == y_test).mean()
print(f"test accuracy: {acc_adam:.4f}")
print()
print(classification_report(y_test, pred, digits=3))
print(pd.DataFrame(confusion_matrix(y_test, pred), index=range(10), columns=range(10)).to_string())

## Change 6 — Does the initialisation actually matter?

Column 1 asserted that He initialisation is important. That is a claim about
this network on this data, so it is worth measuring rather than asserting.

In [ ]:
print("initialisation ablation (100 epochs, no dropout, no weight decay):")
print()
print(f"  {'init':>8}  {'final train loss':>17}  {'test accuracy':>14}  {'dead ReLUs':>11}")
ablation = []
for mode in ["he", "xavier", "small", "zeros"]:
    (W_, b_), (tr_, va_), ep_ = train_network(
        X_train, Y_train, X_val, Y_val, LAYERS,
        epochs=100, lr=1e-3, p_drop=0.0, weight_decay=0.0, patience=1000,
        seed=SEED, verbose=False, label=mode, init_mode=mode,
    )
    p_ = forward(X_test, W_, b_)[0].argmax(axis=1)
    a_ = (p_ == y_test).mean()

    # Fraction of hidden units whose ReLU never activates on the training set
    _, acts_, pre_, _ = forward(X_train, W_, b_)
    dead = float((pre_[0] <= 0).mean())
    ablation.append({"init": mode, "train_loss": tr_[-1], "test_acc": a_, "dead_relu": dead})
    print(f"  {mode:>8}  {tr_[-1]:>17.4f}  {a_:>14.4f}  {dead:>10.1%}")

abl = pd.DataFrame(ablation)
print()
worst_init = abl.loc[abl.test_acc.idxmin()]
print(f"worst initialisation: {worst_init['init']} at {worst_init.test_acc:.4f} "
      f"test accuracy")
print()
if abl.loc[abl["init"] == "zeros", "test_acc"].iloc[0] < 0.2:
    print("Zeros initialisation collapses: every hidden unit receives the same")
    print("zero input, so the gradient w.r.t. those weights is zero and the")
    print("network can never leave that state. Initialisation is not a detail.")
else:
    print("Zeros did not collapse here — worth checking before claiming it does.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
axes[0].bar(abl["init"], abl.test_acc, color="#4C72B0")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("test accuracy")
axes[0].set_title("Test accuracy by initialisation")
axes[0].tick_params(labelsize=8)

axes[1].bar(abl["init"], abl.dead_relu, color="#C44E52")
axes[1].set_ylabel("fraction of dead ReLU units")
axes[1].set_title("Units that never activate")
axes[1].tick_params(labelsize=8)
plt.tight_layout()
plt.show()

## A regularisation sweep, on the validation set only

Column 1 had no regularisation. The strength is a hyperparameter, so it is
chosen on validation data and the test set is evaluated exactly once.

In [ ]:
print("L2 / dropout sweep (selected on VALIDATION loss, never on test):")
print()
print(f"  {'weight_decay':>13}  {'dropout':>8}  {'val loss':>10}  {'epoch':>6}")
sweep = []
for wd in [0.0, 1e-4, 1e-3]:
    for pd_ in [0.0, 0.2, 0.4]:
        (_, _), (tr_, va_), ep_ = train_network(
            X_train, Y_train, X_val, Y_val, LAYERS,
            epochs=80, lr=1e-3, weight_decay=wd, p_drop=pd_,
            patience=20, seed=SEED, verbose=False,
        )
        best_val = min(va_)
        sweep.append({"wd": wd, "dropout": pd_, "val_loss": best_val, "epoch": ep_})
        print(f"  {wd:>13}  {pd_:>8}  {best_val:>10.4f}  {ep_:>6}")

sweep_df = pd.DataFrame(sweep)
best_cfg = sweep_df.loc[sweep_df.val_loss.idxmin()]
print()
print(f"best by validation: weight_decay={best_cfg.wd}, dropout={best_cfg.dropout}, "
      f"val loss {best_cfg.val_loss:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.3))
pivot = sweep_df.pivot(index="wd", columns="dropout", values="val_loss")
im = ax.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c}" for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{i:g}" for i in pivot.index])
ax.set_xlabel("dropout")
ax.set_ylabel("L2 weight decay")
ax.set_title("Validation loss (lower is better)")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the selected configuration on the test set, once.
(W_sel, b_sel), (_, va_sel), ep_sel = train_network(
    X_train, Y_train, X_val, Y_val, LAYERS,
    epochs=80, lr=1e-3, weight_decay=best_cfg.wd, p_drop=best_cfg.dropout,
    patience=20, seed=SEED, verbose=False,
)
pred_sel = forward(X_test, W_sel, b_sel)[0].argmax(axis=1)
acc_sel = (pred_sel == y_test).mean()
print(f"selected config test accuracy: {acc_sel:.4f}")

## Summary

| | Column 1 | Column 2 |
|---|---|---|
| Gradient check | once, by hand | sampled, reusable, run on every fit |
| Dropout path | not checked | checked separately |
| Optimiser | plain SGD, fixed lr | Adam + cosine decay |
| Regularisation | none | L2 and dropout, selected on validation |
| Model selection | none | early stopping on validation |
| Split | train / test | train / validation / test |
| Initialisation | asserted to matter | ablated, including the zeros case |
| Reporting | accuracy | per-class precision, recall, F1 + confusion matrix |

In [ ]:
print("COLUMN 2 RESULTS (from code that ran)")
print(f"  worst gradient error (no dropout) : {worst:.3e}")
print(f"  worst gradient error (dropout)    : {dropout_err:.3e}")
print(f"  best validation loss              : {min(va_hist):.4f}")
print(f"  test accuracy, Adam + early stop   : {acc_adam:.4f}")
print(f"  test accuracy, sweep-selected     : {acc_sel:.4f}")
print(f"  column 1 accuracy (plain SGD)     : 0.9667")
print(f"  zeros-init accuracy (ablation)    : "
      f"{abl.loc[abl['init'] == 'zeros', 'test_acc'].iloc[0]:.4f}, "
      f"{abl.loc[abl['init'] == 'zeros', 'dead_relu'].iloc[0]:.0%} dead ReLUs")
print()
print("Read the accuracy column honestly: Adam plus early stopping reaches")
print(f"{acc_adam:.4f} against column 1's 0.9667, and the regularisation sweep")
print(f"selected the *unregularised* configuration ({acc_sel:.4f}).")
print()
print("This is the result that contradicts the plan. The 2019 version was")
print("already close to the ceiling of what an MLP achieves on 8x8 digits,")
print("and the standard remedies bought almost nothing here — the network is")
print("not overfitting, so regularisation has little to remove. The sweep")
print("found that by measuring rather than assuming, which is the point of")
print("running one.")
print()
print("The load-bearing change is not the accuracy. It is the gradient check:")
print(f"analytic and numerical gradients now agree to {worst:.0e} on every fit,")
print("including the dropout path. That is what would have caught the")
print("ReLU-derivative error in column 1, which is wrong by a factor of 10^8")
print("and produces no exception.")
print()
print("Claiming a 4-point accuracy gain from Adam would be the kind of")
print("improvement this retrospective exists to stop claiming.")